<a href="https://colab.research.google.com/github/Anumay1231/B.S-Detector/blob/main/notebooks/KisanMitra_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KisanMitra 🚜: Tool-Augmented (Agentic) Dealership Assistant
**PSIS Activity 2: Build a Tool-Augmented/Agentic LLM Application**  
Team: I054 Anumay Pandey · I023 Yash Garg · I037 Yash Kothari (B.Tech AI, MPSTME)

An LLM agent for a John Deere tractor dealership. It decides which tool to call: a **weather API** (Open-Meteo), a **market price API** (data.gov.in), the **dealership SQLite database** (inventory, service history, parts), **deterministic finance calculators** (EMI, subsidy, fuel cost) and a **FAISS retriever** over the SMAM subsidy guidelines.

**Run order:** Runtime → Run all, then paste a free Groq API key.

## 1. Install and keys

In [1]:
!pip install -q langchain langchain-core langchain-classic langchain-community langchain-text-splitters langchain-groq langchain-huggingface sentence-transformers faiss-cpu pypdf gradio pandas
# 'requests' is left out on purpose: Colab pins it, and upgrading it prints a dependency conflict warning.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os, sys, getpass
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except Exception:
    pass
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Groq API key (console.groq.com): ')
# Optional: a free data.gov.in key enables live mandi prices. Without it that tool returns TOOL_ERROR on purpose.
# os.environ['DATAGOV_API_KEY'] = '...'
for d in ['src/kisanmitra', 'scripts', 'eval', 'data', 'tests']:
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, 'src'); sys.path.insert(0, 'scripts'); sys.path.insert(0, 'tests')

Groq API key (console.groq.com): ··········


## 2. Application source
Same modules as the GitHub repository, written out so the notebook runs standalone.

In [3]:
%%writefile src/kisanmitra/__init__.py
"""KisanMitra: a tool-augmented (agentic) LLM assistant for tractor dealership and farm operations."""

Writing src/kisanmitra/__init__.py


### Configuration: models, API endpoints, limits

In [4]:
%%writefile src/kisanmitra/config.py
import os

LLM_MODEL = os.getenv("KM_LLM_MODEL", "llama-3.3-70b-versatile")   # fallback only; agent.pick_model() checks what the key can actually call
TEMPERATURE = float(os.getenv("KM_TEMPERATURE", 0))
MAX_ITERATIONS = int(os.getenv("KM_MAX_ITER", 6))          # tool-call loops before the agent must answer
DB_PATH = os.getenv("KM_DB_PATH", "data/dealership.sqlite3")
HTTP_TIMEOUT = int(os.getenv("KM_HTTP_TIMEOUT", 20))
MAX_TOKENS = int(os.getenv("KM_MAX_TOKENS", 600))      # cap answer length to stay inside free-tier TPM
MAX_RETRIES = int(os.getenv("KM_MAX_RETRIES", 8))      # Groq client waits out 429 rate limits
EVAL_PAUSE = float(os.getenv("KM_EVAL_PAUSE", 20))     # seconds between evaluation scenarios
DATAGOV_KEY = os.getenv("DATAGOV_API_KEY", "")

# Free, key-less public APIs
GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"
# data.gov.in daily mandi prices (needs a free API key)
MANDI_URL = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

# Scheme rules used by the retriever tool
SCHEME_PDF_URLS = [
    ("smam_guidelines.pdf", "https://farmech.dac.gov.in/Content/Homepdf/Revised_Operational_Guidelines_SMAM_(2024).pdf"),
    ("smam_guidelines_2020_21.pdf", "https://agrimachinery.nic.in/Files/Guidelines/SMAMGiudeline2020-21.pdf"),
]

Writing src/kisanmitra/config.py


### The tools (LangChain `@tool` with Pydantic argument schemas)
Each tool has a docstring: that text is what the LLM reads when deciding which tool to call. Failures return a `TOOL_ERROR:` string instead of raising, so the agent can recover.

In [5]:
%%writefile src/kisanmitra/tools.py
"""The external tools the agent can call: weather API, market price API, dealership SQL database,
finance calculator and a scheme-rules retriever."""
from __future__ import annotations

import sqlite3
from typing import Optional

import requests
from langchain_core.tools import tool
from pydantic import BaseModel, Field

from . import config

# ---------------------------------------------------------------- helpers

def _db():
    con = sqlite3.connect(config.DB_PATH)
    con.row_factory = sqlite3.Row
    return con


def _rows_to_text(rows, empty: str) -> str:
    if not rows:
        return empty
    return "\n".join("; ".join(f"{k}={r[k]}" for k in r.keys()) for r in rows)


# ---------------------------------------------------------------- 1. weather API

class WeatherArgs(BaseModel):
    location: str = Field(description="Village, town or district name, e.g. 'Dhamtari' or 'Raipur'")
    days: int = Field(default=3, ge=1, le=7, description="Number of forecast days (1-7)")


@tool("get_weather_forecast", args_schema=WeatherArgs)
def get_weather_forecast(location: str, days: int = 3) -> str:
    """Get the daily rain, temperature and wind forecast for an Indian location.
    Use before advising on spraying, harvesting, ploughing or any field operation that depends on rain."""
    try:
        geo = requests.get(config.GEOCODE_URL, params={"name": location, "count": 1, "country": "IN"},
                           timeout=config.HTTP_TIMEOUT).json()
        results = geo.get("results") or []
        if not results:
            return (f"TOOL_ERROR: location '{location}' not found by the geocoding service. "
                    "Ask the user for a nearby district or town name.")
        place = results[0]
        fc = requests.get(config.FORECAST_URL, params={
            "latitude": place["latitude"], "longitude": place["longitude"],
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max",
            "forecast_days": days, "timezone": "auto"}, timeout=config.HTTP_TIMEOUT).json()
        d = fc["daily"]
        lines = [f"Forecast for {place['name']}, {place.get('admin1', '')}:"]
        for i, day in enumerate(d["time"]):
            lines.append(
                f"{day}: rain {d['precipitation_sum'][i]} mm (chance {d['precipitation_probability_max'][i]}%), "
                f"temp {d['temperature_2m_min'][i]}-{d['temperature_2m_max'][i]} C, "
                f"max wind {d['wind_speed_10m_max'][i]} km/h")
        return "\n".join(lines)
    except requests.RequestException as exc:
        return f"TOOL_ERROR: weather service unreachable ({exc.__class__.__name__}). Answer without forecast data and say so."
    except (KeyError, ValueError) as exc:
        return f"TOOL_ERROR: unexpected weather response ({exc}). Do not guess the weather."


# ---------------------------------------------------------------- 2. market price API

class MandiArgs(BaseModel):
    commodity: str = Field(description="Crop name as used in mandi data, e.g. 'Paddy(Dhan)(Common)', 'Wheat', 'Soyabean'")
    state: str = Field(default="Chhattisgarh", description="State name, e.g. 'Chhattisgarh', 'Madhya Pradesh'")


@tool("get_mandi_price", args_schema=MandiArgs)
def get_mandi_price(commodity: str, state: str = "Chhattisgarh") -> str:
    """Get recent mandi (market) prices per quintal for a crop in a state, from the data.gov.in daily price dataset.
    Use when the user asks about selling crops, crop income or whether a purchase is affordable from expected income."""
    if not config.DATAGOV_KEY:
        return ("TOOL_ERROR: no data.gov.in API key configured, so live mandi prices are unavailable. "
                "Tell the user this and continue without price data, or ask them for their expected selling price.")
    try:
        r = requests.get(config.MANDI_URL, params={
            "api-key": config.DATAGOV_KEY, "format": "json", "limit": 10,
            "filters[commodity]": commodity, "filters[state]": state}, timeout=config.HTTP_TIMEOUT)
        r.raise_for_status()
        records = r.json().get("records", [])
        if not records:
            return (f"TOOL_ERROR: no mandi records for commodity='{commodity}' in state='{state}'. "
                    "The commodity name may not match the dataset; ask the user to confirm the crop name.")
        lines = [f"{rec.get('arrival_date')}: {rec.get('market')} ({rec.get('district')}) "
                 f"modal Rs {rec.get('modal_price')}/quintal (min {rec.get('min_price')}, max {rec.get('max_price')})"
                 for rec in records[:5]]
        return f"Mandi prices for {commodity} in {state}:\n" + "\n".join(lines)
    except requests.RequestException as exc:
        return f"TOOL_ERROR: mandi price service unreachable ({exc.__class__.__name__}). Continue without price data."


# ---------------------------------------------------------------- 3. dealership database

class InventoryArgs(BaseModel):
    category: Optional[str] = Field(default=None, description="'tractor', 'implement' or 'harvester'")
    min_hp: Optional[int] = Field(default=None, description="Minimum horsepower required")
    max_price: Optional[int] = Field(default=None, description="Maximum budget in rupees")
    in_stock_only: bool = Field(default=True, description="Only list machines currently in stock")


@tool("search_inventory", args_schema=InventoryArgs)
def search_inventory(category: Optional[str] = None, min_hp: Optional[int] = None,
                     max_price: Optional[int] = None, in_stock_only: bool = True) -> str:
    """Search the dealership's machine inventory (model, horsepower, price, stock, fuel use).
    Always use this instead of guessing which machines or prices the dealership has."""
    sql = "SELECT model, category, hp, price_inr, stock, fuel_lph, notes FROM machines WHERE 1=1"
    params: list = []
    if category:
        sql += " AND category = ?"; params.append(category.lower())
    if min_hp is not None:
        sql += " AND hp >= ?"; params.append(min_hp)
    if max_price is not None:
        sql += " AND price_inr <= ?"; params.append(max_price)
    if in_stock_only:
        sql += " AND stock > 0"
    sql += " ORDER BY price_inr"
    with _db() as con:
        rows = con.execute(sql, params).fetchall()
    return _rows_to_text(rows, "No machines match those filters. Suggest relaxing budget or horsepower.")


class ServiceArgs(BaseModel):
    customer: Optional[str] = Field(default=None, description="Customer name or part of it")
    model: Optional[str] = Field(default=None, description="Machine model, e.g. 'JD 5050D'")
    limit: int = Field(default=5, ge=1, le=20, description="How many recent records to return")


@tool("search_service_records", args_schema=ServiceArgs)
def search_service_records(customer: Optional[str] = None, model: Optional[str] = None, limit: int = 5) -> str:
    """Look up past service jobs (date, engine hours, issue, cost) for a customer or a machine model.
    Use for questions about service history, repeated faults or typical repair cost."""
    sql = ("SELECT service_date, customer, village, model, hours, issue, cost_inr "
           "FROM service_records WHERE 1=1")
    params: list = []
    if customer:
        sql += " AND customer LIKE ?"; params.append(f"%{customer}%")
    if model:
        sql += " AND model LIKE ?"; params.append(f"%{model}%")
    sql += " ORDER BY service_date DESC LIMIT ?"; params.append(limit)
    with _db() as con:
        rows = con.execute(sql, params).fetchall()
    return _rows_to_text(rows, "No service records found for that customer or model.")


class PartArgs(BaseModel):
    query: str = Field(description="Part name or part number, e.g. 'oil filter' or 'AL-172772'")


@tool("check_part_stock", args_schema=PartArgs)
def check_part_stock(query: str) -> str:
    """Check spare-part availability and price at the dealership by part name or part number."""
    with _db() as con:
        rows = con.execute(
            "SELECT part_no, name, model, price_inr, stock FROM parts WHERE name LIKE ? OR part_no LIKE ?",
            (f"%{query}%", f"%{query}%")).fetchall()
    return _rows_to_text(rows, f"No part matching '{query}' in the catalogue.")


# ---------------------------------------------------------------- 4. finance calculator

class FinanceArgs(BaseModel):
    price_inr: float = Field(description="On-road price of the machine in rupees")
    subsidy_percent: float = Field(default=0, ge=0, le=100, description="Subsidy percentage the buyer qualifies for")
    down_payment_inr: float = Field(default=0, ge=0, description="Amount paid upfront")
    annual_rate_percent: float = Field(default=9.5, gt=0, description="Annual loan interest rate")
    years: int = Field(default=5, ge=1, le=15, description="Loan tenure in years")


@tool("calculate_purchase_plan", args_schema=FinanceArgs)
def calculate_purchase_plan(price_inr: float, subsidy_percent: float = 0, down_payment_inr: float = 0,
                            annual_rate_percent: float = 9.5, years: int = 5) -> str:
    """Compute subsidy amount, net cost, loan principal, monthly EMI and total interest for a machine purchase.
    Always use this tool for any EMI, subsidy or affordability arithmetic instead of calculating mentally."""
    subsidy = price_inr * subsidy_percent / 100
    net = price_inr - subsidy
    principal = max(net - down_payment_inr, 0)
    r = annual_rate_percent / 12 / 100
    n = years * 12
    emi = principal * r * (1 + r) ** n / ((1 + r) ** n - 1) if principal > 0 else 0
    total_interest = emi * n - principal
    return (f"price=Rs {price_inr:,.0f}; subsidy({subsidy_percent}%)=Rs {subsidy:,.0f}; net=Rs {net:,.0f}; "
            f"down_payment=Rs {down_payment_inr:,.0f}; loan_principal=Rs {principal:,.0f}; "
            f"emi=Rs {emi:,.0f}/month for {n} months at {annual_rate_percent}%; "
            f"total_interest=Rs {total_interest:,.0f}; total_paid=Rs {principal + total_interest:,.0f}")


class FuelArgs(BaseModel):
    model: str = Field(description="Machine model present in the inventory, e.g. 'JD 5050D'")
    hours: float = Field(gt=0, description="Hours of operation")
    diesel_price_per_litre: float = Field(default=95.0, gt=0, description="Diesel price in rupees per litre")


@tool("estimate_operating_cost", args_schema=FuelArgs)
def estimate_operating_cost(model: str, hours: float, diesel_price_per_litre: float = 95.0) -> str:
    """Estimate diesel consumption and fuel cost for operating a machine for a number of hours.
    Uses the fuel consumption stored for that model in the dealership database."""
    with _db() as con:
        row = con.execute("SELECT model, fuel_lph FROM machines WHERE model LIKE ?", (f"%{model}%",)).fetchone()
    if row is None:
        return f"TOOL_ERROR: model '{model}' not in inventory. Call search_inventory first to get exact model names."
    if not row["fuel_lph"]:
        return f"TOOL_ERROR: {row['model']} is an implement with no engine, so it has no fuel consumption of its own."
    litres = row["fuel_lph"] * hours
    return (f"{row['model']}: {row['fuel_lph']} L/hour x {hours} hours = {litres:.1f} litres; "
            f"fuel cost = Rs {litres * diesel_price_per_litre:,.0f} at Rs {diesel_price_per_litre}/litre")


BASE_TOOLS = [get_weather_forecast, get_mandi_price, search_inventory, search_service_records,
              check_part_stock, calculate_purchase_plan, estimate_operating_cost]

Writing src/kisanmitra/tools.py


### Retriever-as-a-tool (RAG inside the agent)

In [6]:
%%writefile src/kisanmitra/knowledge.py
"""Optional RAG tool: turns the SMAM scheme PDFs into a retriever the agent can call as a tool."""
import os

import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter

from . import config

INDEX_DIR = "faiss_scheme_index"


def download_scheme_pdfs(data_dir: str = "data") -> list[str]:
    os.makedirs(data_dir, exist_ok=True)
    paths = []
    for name, url in config.SCHEME_PDF_URLS:
        path = os.path.join(data_dir, name)
        if not os.path.exists(path):
            try:
                r = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (KisanMitra academic project)"}, timeout=60)
                r.raise_for_status()
                if not r.content.startswith(b"%PDF"):
                    raise ValueError("not a PDF")
                open(path, "wb").write(r.content)
                print(f"saved {name}")
            except Exception as exc:
                print(f"FAILED {name}: {exc}")
                continue
        paths.append(path)
    return paths


def build_scheme_retriever(pdf_paths: list[str], k: int = 3):
    """Index the scheme PDFs and return a VectorStoreRetriever (None if no PDFs are available)."""
    if not pdf_paths:
        return None
    from langchain_huggingface import HuggingFaceEmbeddings
    docs = []
    for path in pdf_paths:
        for page in PyPDFLoader(path).load():
            page.metadata["source_file"] = os.path.basename(path)
            page.metadata["page_no"] = page.metadata.get("page", 0) + 1
            if len(page.page_content.strip()) > 40:
                docs.append(page)
    chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(docs)
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5",
                                       encode_kwargs={"normalize_embeddings": True})
    vs = FAISS.from_documents(chunks, embeddings)
    vs.save_local(INDEX_DIR)
    print(f"Scheme index: {len(chunks)} chunks from {len(pdf_paths)} PDFs")
    return vs.as_retriever(search_type="mmr", search_kwargs={"k": k, "fetch_k": 15})


def make_scheme_tool(retriever):
    """Wrap a retriever as a tool so the agent can decide when scheme rules are needed."""

    @tool("lookup_scheme_rules")
    def lookup_scheme_rules(question: str) -> str:
        """Look up government farm-mechanisation scheme rules (SMAM subsidy percentages, eligibility,
        Custom Hiring Centres) in the official guideline PDFs. Use for any subsidy or eligibility question.
        Returns passages with file and page so they can be cited."""
        docs = retriever.invoke(question)
        if not docs:
            return "TOOL_ERROR: nothing found in the scheme guidelines for that question."
        return "\n\n".join(f"[{d.metadata.get('source_file')} p.{d.metadata.get('page_no')}] {d.page_content[:700]}"
                           for d in docs)

    return lookup_scheme_rules

Writing src/kisanmitra/knowledge.py


### Prompts (`ChatPromptTemplate`, `MessagesPlaceholder`)

In [7]:
%%writefile src/kisanmitra/prompts.py
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM = """You are KisanMitra, an assistant used at a John Deere tractor dealership in Chhattisgarh.
You help staff and farmers with machine selection, purchase planning, service history, spare parts,
field-operation timing and government subsidy rules.

Rules:
1. Never invent prices, stock, horsepower, service history, weather or subsidy rules. Get every such fact
   from a tool. If no tool provides it, say you do not have that information.
2. Do arithmetic (EMI, subsidy, fuel cost) only with calculate_purchase_plan or estimate_operating_cost.
3. You may call several tools in sequence, for example search_inventory to get the exact price and then
   calculate_purchase_plan on that price.
4. If a tool result starts with TOOL_ERROR, do not retry it more than once. Explain the limitation to the
   user in one line and answer with whatever reliable information you do have.
5. Never call the same tool more than twice in one turn. If two calls do not give a complete answer, stop
   calling tools and answer with what you have, saying plainly which part you could not confirm.
6. Quote figures exactly as the tool returned them, in rupees, and name the tool or document you used.
7. Answer in short, practical language. Amounts in Indian rupees, dates as DD Mon YYYY.
8. Refuse politely if the request is outside farm machinery, dealership operations or farming decisions.
9. State as fact only what a tool returned. For wider-market questions (other brands, comparisons, resale
   value, what is "best"), say in one line that you can only speak for the dealership's own systems and offer
   what the tools can show. Do not call extra tools just to satisfy this rule."""

AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

SUMMARY_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Convert the assistant's final answer into the required JSON. Do not add facts that are not in the answer "
     "or the tool log. recommendation must repeat the answer's main advice in at most 60 words.\n\n"
     "Tool log:\n{tool_log}\n\n{format_instructions}"),
    ("human", "User question: {input}\n\nAssistant answer: {output}"),
])

Writing src/kisanmitra/prompts.py


### Structured output schema (`PydanticOutputParser`)

In [8]:
%%writefile src/kisanmitra/schema.py
from typing import List, Literal

from pydantic import BaseModel, Field


class AdviceCard(BaseModel):
    """Structured summary of an agent answer, used by the UI and the evaluation harness."""
    recommendation: str = Field(description="The main advice, at most 60 words")
    key_figures: List[str] = Field(default_factory=list,
                                   description="Exact figures used, e.g. 'EMI Rs 8,611/month', 'price Rs 8,50,000'")
    tools_used: List[str] = Field(default_factory=list, description="Names of the tools that produced those figures")
    data_gaps: str = Field(default="", description="Anything unavailable (failed tool, missing key), else empty")
    confidence: Literal["high", "medium", "low"] = Field(description="high if every figure came from a tool")

Writing src/kisanmitra/schema.py


### Agent assembly: tool-calling agent + `AgentExecutor` + memory + summary chain

In [9]:
%%writefile src/kisanmitra/agent.py
"""Agent assembly: tool-calling agent + executor + conversational memory + structured summary chain."""
from __future__ import annotations

import os

from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory

from . import config
from .prompts import AGENT_PROMPT, SUMMARY_PROMPT
from .schema import AdviceCard
from .tools import BASE_TOOLS

card_parser = PydanticOutputParser(pydantic_object=AdviceCard)


# Groq lists models a key may not actually be allowed to call, so we probe candidates with a
# one-token request and keep the first that answers. Tool calling needs a capable chat model.
PREFERRED_MODELS = [
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b",
    "moonshotai/kimi-k2-instruct",
    "qwen/qwen3-32b",
    "openai/gpt-oss-20b",
    "llama-3.1-8b-instant",
]
SKIP = ("whisper", "tts", "guard", "embed", "vision", "distil", "prompt-guard")


def list_available_models() -> list[str]:
    """Model ids the API lists for this key (not all of them are necessarily callable)."""
    from groq import Groq
    return sorted(m.id for m in Groq().models.list().data)


def model_works(name: str) -> bool:
    """Cheapest possible check that this key can actually call the model."""
    from langchain_groq import ChatGroq
    try:
        ChatGroq(model=name, temperature=0, max_tokens=1).invoke("hi")
        return True
    except Exception as exc:
        print(f"  {name}: unavailable ({str(exc)[:80]})")
        return False


def pick_model(verbose: bool = True) -> str:
    """Return the best model this key can really use. Set KM_LLM_MODEL to override."""
    if os.getenv("KM_LLM_MODEL"):
        return os.environ["KM_LLM_MODEL"]
    try:
        available = list_available_models()
    except Exception as exc:
        print(f"Could not list models ({exc}); trying the preferred list directly")
        available = list(PREFERRED_MODELS)
    candidates = [m for m in PREFERRED_MODELS if m in available]
    candidates += [m for m in available
                   if m not in candidates and not any(s in m.lower() for s in SKIP)]
    if verbose:
        print("Testing which models this key can call:")
    for name in candidates:
        if model_works(name):
            if verbose:
                print(f"Using model: {name}")
            os.environ["KM_LLM_MODEL"] = name      # remember for the rest of the session
            return name
    raise RuntimeError(f"None of these models could be called with this key: {candidates}")


def get_llm(model: str | None = None, temperature: float = config.TEMPERATURE,
            max_tokens: int = config.MAX_TOKENS, max_retries: int = config.MAX_RETRIES):
    """Chat model for the agent. Tool calling is required, so the model must support it.

    max_retries lets the Groq client wait out free-tier rate limits (HTTP 429) instead of failing,
    and max_tokens caps answer length so one question uses fewer tokens per minute.
    """
    from langchain_groq import ChatGroq
    return ChatGroq(model=model or pick_model(), temperature=temperature,
                    max_tokens=max_tokens, max_retries=max_retries)


def build_agent(llm, extra_tools=(), max_iterations: int = config.MAX_ITERATIONS, verbose: bool = True):
    """Return (executor, tools). The executor runs the think -> call tool -> observe loop."""
    tools = list(BASE_TOOLS) + list(extra_tools)
    agent = create_tool_calling_agent(llm, tools, AGENT_PROMPT)
    executor = AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=verbose,
        max_iterations=max_iterations,
        return_intermediate_steps=True,      # needed for evaluation and for the demo
        handle_parsing_errors="Reply with a normal answer or a valid tool call.",
    )
    return executor, tools


def build_conversational_agent(llm, **kwargs):
    executor, tools = build_agent(llm, **kwargs)
    store: dict[str, InMemoryChatMessageHistory] = {}

    def get_history(session_id: str):
        return store.setdefault(session_id, InMemoryChatMessageHistory())

    chain = RunnableWithMessageHistory(
        executor, get_history, input_messages_key="input", history_messages_key="history",
        output_messages_key="output",
    )
    return chain, store, tools


def tool_log(result) -> str:
    """Readable log of every tool call the agent made: name, arguments, observation."""
    lines = []
    for action, observation in result.get("intermediate_steps", []):
        obs = str(observation).replace("\n", " | ")
        lines.append(f"{action.tool}({action.tool_input}) -> {obs[:400]}")
    return "\n".join(lines) if lines else "(no tools were called)"


def summarise(llm, question: str, result) -> AdviceCard:
    """LCEL chain that turns the agent's free-text answer into a validated AdviceCard."""
    chain = SUMMARY_PROMPT.partial(format_instructions=card_parser.get_format_instructions()) | llm | StrOutputParser()
    raw = chain.invoke({"input": question, "output": result["output"], "tool_log": tool_log(result)})
    try:
        return card_parser.parse(raw)
    except OutputParserException:
        fixed = llm.invoke("Return ONLY valid JSON for this schema.\n"
                           f"{card_parser.get_format_instructions()}\n\nText:\n{raw}")
        try:
            return card_parser.parse(fixed.content)
        except OutputParserException:
            return AdviceCard(recommendation=result["output"][:400], key_figures=[], tools_used=[],
                              data_gaps="structured summary unavailable", confidence="low")


STOPPED = "Agent stopped due to max iterations"


def run(chain, llm, question: str, session_id: str = "default", verbose: bool = True):
    result = chain.invoke({"input": question}, config={"configurable": {"session_id": session_id}})
    if STOPPED in result["output"]:
        # The agent used up its tool budget (usually by re-asking one tool). Salvage an answer from
        # the observations it already collected instead of returning the framework's stop message.
        salvage = llm.invoke(
            "Answer the user's question using ONLY the tool results below. Quote figures exactly and "
            "say plainly what could not be confirmed.\n\n"
            f"Question: {question}\n\nTool results:\n{tool_log(result)}"
        )
        result["output"] = salvage.content.strip() + "\n\n(Answer assembled after the tool-call limit was reached.)"
    card = summarise(llm, question, result)
    if verbose:
        print(f"\nQ: {question}\nA: {result['output']}")
        print("Tools called:\n  " + tool_log(result).replace("\n", "\n  "))
        print(f"Card: {card.model_dump()}")
    return {"result": result, "card": card, "tool_log": tool_log(result)}

Writing src/kisanmitra/agent.py


### Dealership database seed

In [10]:
%%writefile scripts/seed_db.py
"""Create data/dealership.sqlite3 with a small synthetic dealership dataset."""
import os
import random
import sqlite3
import sys
from datetime import date, timedelta

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from kisanmitra import config  # noqa: E402

MACHINES = [
    # model, category, hp, price_inr, stock, fuel_lph, notes
    ("JD 5050D", "tractor", 50, 850000, 4, 4.2, "2WD utility tractor, 8F+4R gears"),
    ("JD 5105", "tractor", 40, 720000, 2, 3.6, "2WD, popular for rotavator work"),
    ("JD 5310 4WD", "tractor", 55, 1180000, 1, 4.8, "4WD, suited to wet paddy fields"),
    ("JD 3028EN", "tractor", 28, 610000, 3, 2.4, "Narrow orchard tractor"),
    ("JD W70 Harvester", "harvester", 76, 2650000, 1, 9.5, "Self-propelled combine harvester"),
    ("Rotavator RT-150", "implement", 0, 78000, 6, 0.0, "1.5 m rotavator, needs 45+ hp"),
    ("MB Plough 2B", "implement", 0, 42000, 5, 0.0, "2-bottom mould board plough"),
    ("Seed Drill SD-11", "implement", 0, 65000, 2, 0.0, "11-tyne seed cum fertiliser drill"),
    ("Baler RB-120", "implement", 0, 890000, 0, 0.0, "Round baler, out of stock"),
]
PARTS = [
    ("AL-172772", "Engine oil filter", "JD 5050D", 640, 18),
    ("RE-509672", "Fuel filter", "JD 5050D", 1180, 6),
    ("AL-150288", "Air filter element", "JD 5105", 1450, 0),
    ("RE-45864", "Hydraulic filter", "JD 5310 4WD", 2260, 4),
    ("TY-6382", "Front tyre 6.00-16", "JD 5105", 5400, 8),
]
ISSUES = ["250-hour service", "hydraulic lift slow", "clutch adjustment", "overheating",
          "starter motor replacement", "500-hour service", "oil leak from rear axle"]
CUSTOMERS = ["Ramesh Sahu", "Devendra Patel", "Sunita Verma", "Gopal Yadav", "Mahesh Chandrakar"]
VILLAGES = ["Dhamtari", "Rajnandgaon", "Durg", "Bemetara", "Mahasamund"]


def main():
    os.makedirs(os.path.dirname(config.DB_PATH), exist_ok=True)
    if os.path.exists(config.DB_PATH):
        os.remove(config.DB_PATH)
    con = sqlite3.connect(config.DB_PATH)
    c = con.cursor()
    c.execute("""CREATE TABLE machines (model TEXT PRIMARY KEY, category TEXT, hp INTEGER,
                 price_inr INTEGER, stock INTEGER, fuel_lph REAL, notes TEXT)""")
    c.execute("""CREATE TABLE parts (part_no TEXT PRIMARY KEY, name TEXT, model TEXT,
                 price_inr INTEGER, stock INTEGER)""")
    c.execute("""CREATE TABLE service_records (id INTEGER PRIMARY KEY, customer TEXT, village TEXT,
                 model TEXT, service_date TEXT, hours INTEGER, issue TEXT, cost_inr INTEGER)""")
    c.executemany("INSERT INTO machines VALUES (?,?,?,?,?,?,?)", MACHINES)
    c.executemany("INSERT INTO parts VALUES (?,?,?,?,?)", PARTS)

    random.seed(7)
    rows = []
    tractors = [m[0] for m in MACHINES if m[1] == "tractor"]
    for i in range(1, 41):
        cust = random.choice(CUSTOMERS)
        rows.append((i, cust, random.choice(VILLAGES), random.choice(tractors),
                     (date(2026, 9, 1) - timedelta(days=random.randint(5, 700))).isoformat(),
                     random.choice([250, 500, 750, 1000, 1250]), random.choice(ISSUES),
                     random.randint(1200, 18000)))
    c.executemany("INSERT INTO service_records VALUES (?,?,?,?,?,?,?,?)", rows)
    con.commit()
    print(f"Seeded {config.DB_PATH}: {len(MACHINES)} machines, {len(PARTS)} parts, {len(rows)} service records")
    con.close()


if __name__ == "__main__":
    main()

Writing scripts/seed_db.py


### Evaluation harness

In [11]:
%%writefile scripts/evaluate.py
"""Run the agent on the test scenarios and score tool selection, arithmetic and error handling."""
import ast
import json
import os
import re
import sys
import time

import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from kisanmitra import config  # noqa: E402
from kisanmitra.agent import build_conversational_agent, get_llm, run  # noqa: E402


def run_with_backoff(chain, llm, question, session_id, attempts: int = 4):
    """Free Groq keys have a low tokens-per-minute limit. On a 429, wait and try again."""
    for attempt in range(attempts):
        try:
            return run(chain, llm, question, session_id=session_id, verbose=True)
        except Exception as exc:
            text = str(exc)
            if "rate_limit" not in text and "429" not in text:
                raise
            wait = 30 * (attempt + 1)
            m = re.search(r"try again in ([\d.]+)s", text)
            if m:
                wait = max(float(m.group(1)) + 5, 20)
            print(f"  rate limited, waiting {wait:.0f}s (attempt {attempt + 1}/{attempts})")
            time.sleep(wait)
    raise RuntimeError("Still rate limited after several retries; wait a minute and re-run.")


def emi(principal, annual_rate_percent, years):
    r = annual_rate_percent / 12 / 100
    n = years * 12
    return principal * r * (1 + r) ** n / ((1 + r) ** n - 1)


def verify_calculations(tool_log: str) -> str:
    """Independently re-compute every EMI the calculator tool returned, using the arguments the
    agent actually passed. This checks the tool's arithmetic without assuming which machine it chose."""
    checks, bad = 0, []
    for line in tool_log.splitlines():
        if not line.startswith("calculate_purchase_plan("):
            continue
        args_text = line[len("calculate_purchase_plan("):line.rindex(") ->")]
        try:
            args = ast.literal_eval(args_text)
            quoted = int(re.search(r"emi=Rs ([\d,]+)", line).group(1).replace(",", ""))
        except (ValueError, SyntaxError, AttributeError):
            continue
        principal = (args["price_inr"] * (1 - args.get("subsidy_percent", 0) / 100)
                     - args.get("down_payment_inr", 0))
        expected = round(emi(principal, args.get("annual_rate_percent", 9.5), args.get("years", 5)))
        checks += 1
        if abs(expected - quoted) > 2:
            bad.append(f"{quoted} vs expected {expected}")
    if not checks:
        return ""
    return f"verified ({checks} calculation(s))" if not bad else "mismatch: " + "; ".join(bad)


def evaluate(chain, llm, scenarios, pause: float = config.EVAL_PAUSE) -> pd.DataFrame:
    rows = []
    for s in scenarios:
        t0 = time.time()
        print(f"\n[{s['id']}] {s['category']}")
        out = run_with_backoff(chain, llm, s["question"], s.get("session", s["id"]))
        latency = round(time.time() - t0, 2)
        steps = out["result"].get("intermediate_steps", [])
        called = [a.tool for a, _ in steps]
        observations = " ".join(str(o) for _, o in steps)
        expected = s.get("expected_tools", [])
        rows.append({
            "id": s["id"], "category": s["category"], "question": s["question"],
            "tools_called": ", ".join(called) or "(none)",
            "expected_tools": ", ".join(expected) or "(none)",
            "tool_selection_ok": set(expected).issubset(set(called)) if expected else called == [],
            "steps": len(steps),
            "tool_error_seen": "TOOL_ERROR" in observations,
            "error_handled_ok": (("TOOL_ERROR" in observations) == s.get("expect_tool_error", False)),
            "answer": out["result"]["output"],
            "key_figures": "; ".join(out["card"].key_figures),
            "confidence": out["card"].confidence,
            "data_gaps": out["card"].data_gaps,
            "latency_s": latency,
        })
        print("-" * 95)
        time.sleep(pause)   # stay under the free-tier tokens-per-minute limit
    df = pd.DataFrame(rows)
    df["arithmetic_check"] = df["tool_log"].apply(verify_calculations)
    return df


def summarise(df: pd.DataFrame):
    print(f"Tool selection correct : {df['tool_selection_ok'].mean():.0%} ({df['tool_selection_ok'].sum()}/{len(df)})")
    print(f"Error handling correct : {df['error_handled_ok'].mean():.0%}")
    print(f"Mean tool calls        : {df['steps'].mean():.1f}")
    print(f"Mean latency           : {df['latency_s'].mean():.1f}s")
    arith = [a for a in df["arithmetic_check"] if a]
    if arith:
        ok = sum(a.startswith("verified") for a in arith)
        print(f"Independent EMI check  : {ok}/{len(arith)} scenarios verified" +
              ("" if ok == len(arith) else " -> " + "; ".join(a for a in arith if not a.startswith('verified'))))


if __name__ == "__main__":
    scenarios = json.load(open(os.path.join(os.path.dirname(__file__), "..", "eval", "test_scenarios.json")))
    llm = get_llm()
    extra = []
    try:                                    # scheme retriever tool, if the PDFs and model are available
        from kisanmitra.knowledge import build_scheme_retriever, download_scheme_pdfs, make_scheme_tool
        retriever = build_scheme_retriever(download_scheme_pdfs())
        if retriever:
            extra = [make_scheme_tool(retriever)]
    except Exception as exc:
        print(f"Scheme tool unavailable: {exc}")
    chain, _, _ = build_conversational_agent(llm, extra_tools=extra)
    df = evaluate(chain, llm, scenarios)
    os.makedirs("eval", exist_ok=True)
    df.to_csv("eval/results.csv", index=False)
    summarise(df)

Writing scripts/evaluate.py


### Scripted model used by the offline tests (no API key needed)

In [12]:
%%writefile tests/fake_llm.py
"""A scripted tool-calling chat model so the agent loop can be tested without an API key."""
import json
from typing import Any, List

from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult


class ScriptedToolLLM(BaseChatModel):
    """Emits the queued tool calls one per turn, then a final answer; JSON when asked to summarise."""
    script: List[Any] = []
    step: int = 0

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, stop=None, **kwargs) -> ChatResult:
        text = " ".join(str(m.content) for m in messages)
        if "format_instructions" in text or "required JSON" in text or "ONLY valid JSON" in text:
            msg = AIMessage(content=json.dumps({
                "recommendation": "Scripted summary", "key_figures": ["EMI Rs 8,611/month"],
                "tools_used": ["search_inventory", "calculate_purchase_plan"], "data_gaps": "",
                "confidence": "high"}))
            return ChatResult(generations=[ChatGeneration(message=msg)])
        if self.step < len(self.script):
            item = self.script[self.step]
            self.step += 1
            if isinstance(item, tuple):
                name, args = item
                msg = AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": f"call_{self.step}"}])
            else:
                msg = AIMessage(content=item)
            return ChatResult(generations=[ChatGeneration(message=msg)])
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content="Final scripted answer."))])

Writing tests/fake_llm.py


### Offline tests of the agent loop

In [13]:
%%writefile tests/test_agent_offline.py
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
sys.path.insert(0, os.path.dirname(__file__))
os.chdir(os.path.join(os.path.dirname(__file__), ".."))

from fake_llm import ScriptedToolLLM
from kisanmitra.agent import build_conversational_agent, run

def test_multi_tool_flow():
    llm = ScriptedToolLLM(script=[
        ("search_inventory", {"category": "tractor", "max_price": 900000}),
        ("calculate_purchase_plan", {"price_inr": 850000, "subsidy_percent": 40, "down_payment_inr": 100000}),
        "The JD 5050D at Rs 8,50,000 fits: after 40% subsidy and Rs 1,00,000 down payment the EMI is Rs 8,611/month.",
    ])
    chain, store, tools = build_conversational_agent(llm, verbose=False)
    out = run(chain, llm, "Which tractor under 9 lakh suits me and what would the EMI be?", "t1", verbose=False)
    steps = out["result"]["intermediate_steps"]
    assert [s[0].tool for s in steps] == ["search_inventory", "calculate_purchase_plan"], steps
    assert "8,611" in out["result"]["output"]
    assert out["card"].confidence == "high"
    assert len(store["t1"].messages) == 2
    print("PASS multi-tool flow:", [s[0].tool for s in steps])

def test_tool_error_handling():
    llm = ScriptedToolLLM(script=[
        ("get_weather_forecast", {"location": "Nowhereville", "days": 3}),
        "I could not fetch the forecast, so I cannot advise on spraying today.",
    ])
    chain, _, _ = build_conversational_agent(llm, verbose=False)
    out = run(chain, llm, "Can I spray tomorrow in Nowhereville?", "t2", verbose=False)
    obs = out["result"]["intermediate_steps"][0][1]
    assert "TOOL_ERROR" in obs, obs
    assert "could not fetch" in out["result"]["output"]
    print("PASS tool-error handling:", obs[:60])

def test_no_tool_needed():
    llm = ScriptedToolLLM(script=["I can help with machines, service, subsidies and field timing."])
    chain, _, _ = build_conversational_agent(llm, verbose=False)
    out = run(chain, llm, "What can you do?", "t3", verbose=False)
    assert out["result"]["intermediate_steps"] == []
    print("PASS no-tool path")

def test_max_iterations_guard():
    llm = ScriptedToolLLM(script=[("check_part_stock", {"query": "filter"})] * 20)
    chain, _, _ = build_conversational_agent(llm, verbose=False, max_iterations=3)
    out = run(chain, llm, "loop test", "t4", verbose=False)
    assert len(out["result"]["intermediate_steps"]) <= 3
    print("PASS max-iteration guard:", len(out["result"]["intermediate_steps"]), "steps")

if __name__ == "__main__":
    test_multi_tool_flow(); test_tool_error_handling(); test_no_tool_needed(); test_max_iterations_guard()
    print("\nAll offline agent tests passed.")

Writing tests/test_agent_offline.py


### Test scenarios

In [14]:
%%writefile eval/test_scenarios.json
[
  {"id": "S1", "category": "single tool (database)", "question": "Which tractors do you have in stock under 9 lakh rupees?", "expected_tools": ["search_inventory"], "session": "e1"},
  {"id": "S2", "category": "multi-step (database then calculator)", "question": "I am a small farmer with 1 lakh down payment. Suggest a tractor in stock under 9 lakh and tell me the EMI over 5 years with 40% subsidy.", "expected_tools": ["search_inventory", "calculate_purchase_plan"], "session": "e2"},
  {"id": "S3", "category": "memory follow-up", "question": "What if I take it for 7 years instead?", "expected_tools": ["calculate_purchase_plan"], "session": "e2"},
  {"id": "S4", "category": "external API (weather)", "question": "I want to spray my paddy field near Dhamtari. Is the weather suitable in the next 3 days?", "expected_tools": ["get_weather_forecast"], "session": "e3"},
  {"id": "S5", "category": "database (service history)", "question": "Show the recent service jobs for Ramesh Sahu and what they cost.", "expected_tools": ["search_service_records"], "session": "e4"},
  {"id": "S6", "category": "calculator (fuel)", "question": "How much diesel cost should I expect if I run the JD 5050D for 300 hours at Rs 92 per litre?", "expected_tools": ["estimate_operating_cost"], "session": "e5"},
  {"id": "S7", "category": "tool failure (expected)", "question": "What is the mandi price of paddy in Chhattisgarh right now?", "expected_tools": ["get_mandi_price"], "session": "e6", "expect_tool_error": true},
  {"id": "S8", "category": "scheme rules (RAG tool)", "question": "What subsidy percentage can an SC farmer get under SMAM for farm machinery?", "expected_tools": ["lookup_scheme_rules"], "session": "e7"},
  {"id": "S9", "category": "out of scope (should refuse)", "question": "Write me a poem about my village festival.", "expected_tools": [], "session": "e8", "expect_refusal": true}
]

Writing eval/test_scenarios.json


## 3. Build the dealership database

In [15]:
!python scripts/seed_db.py
import sqlite3, pandas as pd
con = sqlite3.connect('data/dealership.sqlite3')
display(pd.read_sql('SELECT * FROM machines', con))
display(pd.read_sql('SELECT * FROM service_records LIMIT 5', con))

Seeded data/dealership.sqlite3: 9 machines, 5 parts, 40 service records


,model,category,hp,price_inr,stock,fuel_lph,notes
0,JD 5050D,tractor,50,850000,4,4.2,"2WD utility tractor, 8F+4R gears"
1,JD 5105,tractor,40,720000,2,3.6,"2WD, popular for rotavator work"
2,JD 5310 4WD,tractor,55,1180000,1,4.8,"4WD, suited to wet paddy fields"
3,JD 3028EN,tractor,28,610000,3,2.4,Narrow orchard tractor
4,JD W70 Harvester,harvester,76,2650000,1,9.5,Self-propelled combine harvester
5,Rotavator RT-150,implement,0,78000,6,0.0,"1.5 m rotavator, needs 45+ hp"
6,MB Plough 2B,implement,0,42000,5,0.0,2-bottom mould board plough
7,Seed Drill SD-11,implement,0,65000,2,0.0,11-tyne seed cum fertiliser drill
8,Baler RB-120,implement,0,890000,0,0.0,"Round baler, out of stock"


,id,customer,village,model,service_date,hours,issue,cost_inr
0,1,Sunita Verma,Rajnandgaon,JD 3028EN,2024-10-30,250,250-hour service,4284
1,2,Sunita Verma,Mahasamund,JD 5050D,2025-03-26,500,250-hour service,4016
2,3,Gopal Yadav,Bemetara,JD 5050D,2025-12-24,250,starter motor replacement,15110
3,4,Ramesh Sahu,Mahasamund,JD 5050D,2026-01-11,1250,250-hour service,14198
4,5,Ramesh Sahu,Rajnandgaon,JD 5050D,2025-02-03,500,clutch adjustment,14934


## 4. Offline tests (no API key used)
Checks the agent loop, tool-error handling and the iteration guard with a scripted model.

In [16]:
!python tests/test_agent_offline.py

/content/tests/test_agent_offline.py:15: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chain, store, tools = build_conversational_agent(llm, verbose=False)
PASS multi-tool flow: ['search_inventory', 'calculate_purchase_plan']
/content/tests/test_agent_offline.py:29: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chain, _, _ = build_conversational_agent(llm, verbose=False)
PASS tool-error handling: TOOL_ERROR: location 'Nowhereville' not found by the geocodi
/content/tests/test_agent_offline.py:38: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chain, _, _ = build_conversational_agent(llm, verbose=False)
PASS no-tool path
/content/tests/test_agent_offline.py:45: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence inste

## 5. Try each tool directly
Proves the tools work before the LLM is involved.

In [17]:
from kisanmitra.tools import BASE_TOOLS, search_inventory, calculate_purchase_plan, get_weather_forecast, estimate_operating_cost
print([t.name for t in BASE_TOOLS], '\n')
print(search_inventory.invoke({'category': 'tractor', 'max_price': 900000}), '\n')
print(calculate_purchase_plan.invoke({'price_inr': 850000, 'subsidy_percent': 40, 'down_payment_inr': 100000}), '\n')
print(estimate_operating_cost.invoke({'model': 'JD 5050D', 'hours': 300, 'diesel_price_per_litre': 92}), '\n')
print(get_weather_forecast.invoke({'location': 'Dhamtari', 'days': 3}))

['get_weather_forecast', 'get_mandi_price', 'search_inventory', 'search_service_records', 'check_part_stock', 'calculate_purchase_plan', 'estimate_operating_cost'] 

model=JD 3028EN; category=tractor; hp=28; price_inr=610000; stock=3; fuel_lph=2.4; notes=Narrow orchard tractor
model=JD 5105; category=tractor; hp=40; price_inr=720000; stock=2; fuel_lph=3.6; notes=2WD, popular for rotavator work
model=JD 5050D; category=tractor; hp=50; price_inr=850000; stock=4; fuel_lph=4.2; notes=2WD utility tractor, 8F+4R gears 

price=Rs 850,000; subsidy(40.0%)=Rs 340,000; net=Rs 510,000; down_payment=Rs 100,000; loan_principal=Rs 410,000; emi=Rs 8,611/month for 60 months at 9.5%; total_interest=Rs 106,646; total_paid=Rs 516,646 

JD 5050D: 4.2 L/hour x 300.0 hours = 1260.0 litres; fuel cost = Rs 115,920 at Rs 92.0/litre 

Forecast for Dhamtari, Chhattisgarh:
2026-09-24: rain 91.5 mm (chance 100%), temp 24.0-26.0 C, max wind 31.9 km/h
2026-09-25: rain 64.7 mm (chance 100%), temp 23.5-25.7 C, max wind

## 6. Scheme retriever tool (optional)
Downloads the SMAM guideline PDFs and indexes them. If a download fails, the agent simply runs without this tool.

In [18]:
extra_tools = []
try:
    from kisanmitra.knowledge import download_scheme_pdfs, build_scheme_retriever, make_scheme_tool
    retriever = build_scheme_retriever(download_scheme_pdfs())
    if retriever:
        extra_tools = [make_scheme_tool(retriever)]
except Exception as exc:
    print('Scheme tool unavailable:', exc)
print('extra tools:', [t.name for t in extra_tools])

FAILED smam_guidelines.pdf: 403 Client Error: Forbidden for url: https://farmech.dac.gov.in/Content/Homepdf/Revised_Operational_Guidelines_SMAM_(2024).pdf
saved smam_guidelines_2020_21.pdf


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Scheme index: 221 chunks from 1 PDFs
extra tools: ['lookup_scheme_rules']


## 7. Check which models this key can use, then build the agent
Groq changes model availability per key, so the agent asks the API which models are allowed and picks the best one it can call. Override with `os.environ['KM_LLM_MODEL'] = '<id>'` if you want a specific one.

In [19]:
from kisanmitra.agent import list_available_models, pick_model
print('Models listed for this key:')
for m in list_available_models():
    print(' ', m)

# Listing a model does not guarantee access, so probe with a one-token call:
MODEL = pick_model()

Models listed for this key:
  allam-2-7b
  canopylabs/orpheus-arabic-saudi
  canopylabs/orpheus-v1-english
  meta-llama/llama-prompt-guard-2-22m
  meta-llama/llama-prompt-guard-2-86m
  openai/gpt-oss-120b
  openai/gpt-oss-20b
  openai/gpt-oss-safeguard-20b
  qwen/qwen3.8-27b
  whisper-large-v3
  whisper-large-v3-turbo
Testing which models this key can call:
Using model: openai/gpt-oss-120b


In [20]:
from kisanmitra.agent import get_llm, build_conversational_agent, run
llm = get_llm(MODEL)
chain, store, tools = build_conversational_agent(llm, extra_tools=extra_tools, verbose=True)
print('Tools bound to the agent:', [t.name for t in tools])

Tools bound to the agent: ['get_weather_forecast', 'get_mandi_price', 'search_inventory', 'search_service_records', 'check_part_stock', 'calculate_purchase_plan', 'estimate_operating_cost', 'lookup_scheme_rules']


/tmp/ipykernel_5031/4168471607.py:3: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chain, store, tools = build_conversational_agent(llm, extra_tools=extra_tools, verbose=True)


## 8. Demonstration: single tool, multi-step reasoning, memory

In [21]:
_ = run(chain, llm, 'Which tractors under 9 lakh do we have in stock?', session_id='demo')



> Entering new AgentExecutor chain...

Invoking: `search_inventory` with `{'category': 'tractor', 'in_stock_only': True, 'max_price': 900000}`


model=JD 3028EN; category=tractor; hp=28; price_inr=610000; stock=3; fuel_lph=2.4; notes=Narrow orchard tractor
model=JD 5105; category=tractor; hp=40; price_inr=720000; stock=2; fuel_lph=3.6; notes=2WD, popular for rotavator work
model=JD 5050D; category=tractor; hp=50; price_inr=850000; stock=4; fuel_lph=4.2; notes=2WD utility tractor, 8F+4R gears**Tractors in stock under ₹9 lakh (search_inventory):**

| Model | Horsepower | On‑road price | Stock |
|-------|------------|---------------|-------|
| JD 3028EN | 28 hp | ₹610,000 | 3 |
| JD 5105 | 40 hp | ₹720,000 | 2 |
| JD 5050D | 50 hp | ₹850,000 | 4 |

All three are currently available at the dealership.

> Finished chain.

Q: Which tractors under 9 lakh do we have in stock?
A: **Tractors in stock under ₹9 lakh (search_inventory):**

| Model | Horsepower | On‑road price | Stock |
|-------|-

In [22]:
_ = run(chain, llm, 'I can pay 1 lakh upfront and I qualify for 40% subsidy. Recommend one of those and give me the 5-year EMI.', session_id='demo')



> Entering new AgentExecutor chain...

Invoking: `calculate_purchase_plan` with `{'annual_rate_percent': 9.5, 'down_payment_inr': 100000, 'price_inr': 850000, 'subsidy_percent': 40, 'years': 5}`


price=Rs 850,000; subsidy(40.0%)=Rs 340,000; net=Rs 510,000; down_payment=Rs 100,000; loan_principal=Rs 410,000; emi=Rs 8,611/month for 60 months at 9.5%; total_interest=Rs 106,646; total_paid=Rs 516,646
Invoking: `calculate_purchase_plan` with `{'annual_rate_percent': 9.5, 'down_payment_inr': 100000, 'price_inr': 720000, 'subsidy_percent': 40, 'years': 5}`


price=Rs 720,000; subsidy(40.0%)=Rs 288,000; net=Rs 432,000; down_payment=Rs 100,000; loan_principal=Rs 332,000; emi=Rs 6,973/month for 60 months at 9.5%; total_interest=Rs 86,357; total_paid=Rs 418,357**Recommendation:** **JD 5105** (40 hp)  

- **On‑road price:** ₹720,000 (search_inventory)  
- **Subsidy (40 %):** ₹288,000  
- **Net price after subsidy:** ₹432,000  
- **Down‑payment:** ₹100,000 (your upfront amount)  
- **Loan princi

In [23]:
_ = run(chain, llm, 'What if I take it for 7 years instead?', session_id='demo')   # needs memory



> Entering new AgentExecutor chain...

Invoking: `calculate_purchase_plan` with `{'annual_rate_percent': 9.5, 'down_payment_inr': 100000, 'price_inr': 720000, 'subsidy_percent': 40, 'years': 7}`


price=Rs 720,000; subsidy(40.0%)=Rs 288,000; net=Rs 432,000; down_payment=Rs 100,000; loan_principal=Rs 332,000; emi=Rs 5,426/month for 84 months at 9.5%; total_interest=Rs 123,801; total_paid=Rs 455,801**JD 5105 – 7‑year loan (calculate_purchase_plan)**  

- **On‑road price:** ₹720,000 (search_inventory)  
- **Subsidy (40 %):** ₹288,000  
- **Net price after subsidy:** ₹432,000  
- **Down‑payment:** ₹100,000 (your upfront amount)  
- **Loan principal:** ₹332,000  

**EMI for 7 years (84 months) at 9.5 %:** **₹5,426 per month**  

- **Total interest over 7 years:** ₹123,801  
- **Total amount paid (down‑payment + EMIs):** ₹455,801  

The longer tenure reduces the monthly EMI compared with a 5‑year plan, making the payment easier to manage while the total interest cost rises.

> Finished cha

## 9. Evaluation on 9 scenarios
Scores tool-selection correctness, error handling, number of tool calls and latency, and independently re-computes the EMI in Python to verify the calculator.

In [25]:
%%writefile scripts/evaluate.py
"""Run the agent on the test scenarios and score tool selection, arithmetic and error handling."""
import ast
import json
import os
import re
import sys
import time

import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from kisanmitra import config  # noqa: E402
from kisanmitra.agent import build_conversational_agent, get_llm, run  # noqa: E402


def run_with_backoff(chain, llm, question, session_id, attempts: int = 4):
    """Free Groq keys have a low tokens-per-minute limit. On a 429, wait and try again."""
    for attempt in range(attempts):
        try:
            return run(chain, llm, question, session_id=session_id, verbose=True)
        except Exception as exc:
            text = str(exc)
            if "rate_limit" not in text and "429" not in text:
                raise
            wait = 30 * (attempt + 1)
            m = re.search(r"try again in ([\d.]+)s", text)
            if m:
                wait = max(float(m.group(1)) + 5, 20)
            print(f"  rate limited, waiting {wait:.0f}s (attempt {attempt + 1}/{attempts})")
            time.sleep(wait)
    raise RuntimeError("Still rate limited after several retries; wait a minute and re-run.")


def emi(principal, annual_rate_percent, years):
    r = annual_rate_percent / 12 / 100
    n = years * 12
    return principal * r * (1 + r) ** n / ((1 + r) ** n - 1)


def verify_calculations(tool_log: str) -> str:
    """Independently re-compute every EMI the calculator tool returned, using the arguments the
    agent actually passed. This checks the tool's arithmetic without assuming which machine it chose."""
    checks, bad = 0, []
    for line in tool_log.splitlines():
        if not line.startswith("calculate_purchase_plan("):
            continue
        args_text = line[len("calculate_purchase_plan("):line.rindex(") ->")]
        try:
            args = ast.literal_eval(args_text)
            quoted = int(re.search(r"emi=Rs ([\d,]+)", line).group(1).replace(",", ""))
        except (ValueError, SyntaxError, AttributeError):
            continue
        principal = (args["price_inr"] * (1 - args.get("subsidy_percent", 0) / 100)
                     - args.get("down_payment_inr", 0))
        expected = round(emi(principal, args.get("annual_rate_percent", 9.5), args.get("years", 5)))
        checks += 1
        if abs(expected - quoted) > 2:
            bad.append(f"{quoted} vs expected {expected}")
    if not checks:
        return ""
    return f"verified ({checks} calculation(s))" if not bad else "mismatch: " + "; ".join(bad)


def evaluate(chain, llm, scenarios, pause: float = config.EVAL_PAUSE) -> pd.DataFrame:
    rows = []
    for s in scenarios:
        t0 = time.time()
        print(f"\n[{s['id']}] {s['category']}")
        out = run_with_backoff(chain, llm, s["question"], s.get("session", s["id"]))
        latency = round(time.time() - t0, 2)
        steps = out["result"].get("intermediate_steps", [])
        called = [a.tool for a, _ in steps]
        observations = " ".join(str(o) for _, o in steps)
        expected = s.get("expected_tools", [])
        rows.append({
            "id": s["id"], "category": s["category"], "question": s["question"],
            "tools_called": ", ".join(called) or "(none)",
            "expected_tools": ", ".join(expected) or "(none)",
            "tool_selection_ok": set(expected).issubset(set(called)) if expected else called == [],
            "steps": len(steps),
            "tool_error_seen": "TOOL_ERROR" in observations,
            "error_handled_ok": (("TOOL_ERROR" in observations) == s.get("expect_tool_error", False)),
            "answer": out["result"]["output"],
            "key_figures": "; ".join(out["card"].key_figures),
            "confidence": out["card"].confidence,
            "data_gaps": out["card"].data_gaps,
            "latency_s": latency,
            "tool_log": out["tool_log"],
        })
        print("-" * 95)
        time.sleep(pause)   # stay under the free-tier tokens-per-minute limit
    df = pd.DataFrame(rows)
    df["arithmetic_check"] = df["tool_log"].apply(verify_calculations)
    return df


def summarise(df: pd.DataFrame):
    print(f"Tool selection correct : {df['tool_selection_ok'].mean():.0%} ({df['tool_selection_ok'].sum()}/{len(df)})")
    print(f"Error handling correct : {df['error_handled_ok'].mean():.0%}")
    print(f"Mean tool calls        : {df['steps'].mean():.1f}")
    print(f"Mean latency           : {df['latency_s'].mean():.1f}s")
    arith = [a for a in df["arithmetic_check"] if a]
    if arith:
        ok = sum(a.startswith("verified") for a in arith)
        print(f"Independent EMI check  : {ok}/{len(arith)} scenarios verified" +
              ("" if ok == len(arith) else " -> " + "; ".join(a for a in arith if not a.startswith('verified'))))

Overwriting scripts/evaluate.py


In [27]:
import json, importlib, evaluate as ev
ev = importlib.reload(ev)
from kisanmitra.agent import build_conversational_agent

# fresh chain = empty memory, so each scenario starts clean
eval_chain, eval_store, _ = build_conversational_agent(llm, extra_tools=extra_tools, verbose=True)

scenarios = json.load(open('eval/test_scenarios.json'))
results = ev.evaluate(eval_chain, llm, scenarios, pause=20)
results.to_csv('eval/results.csv', index=False)
ev.summarise(results)
results[['id','category','tools_called','expected_tools','tool_selection_ok','steps','error_handled_ok','latency_s']]

/tmp/ipykernel_5031/2709393905.py:6: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  eval_chain, eval_store, _ = build_conversational_agent(llm, extra_tools=extra_tools, verbose=True)



[S1] single tool (database)


> Entering new AgentExecutor chain...

Invoking: `search_inventory` with `{'category': 'tractor', 'in_stock_only': True, 'max_price': 900000}`


model=JD 3028EN; category=tractor; hp=28; price_inr=610000; stock=3; fuel_lph=2.4; notes=Narrow orchard tractor
model=JD 5105; category=tractor; hp=40; price_inr=720000; stock=2; fuel_lph=3.6; notes=2WD, popular for rotavator work
model=JD 5050D; category=tractor; hp=50; price_inr=850000; stock=4; fuel_lph=4.2; notes=2WD utility tractor, 8F+4R gears**Tractors in stock ≤ ₹9  lakh (as of today)**  

| Model | Horsepower | On‑road price | Units in stock |
|-------|------------|---------------|----------------|
| JD 3028EN | 28 hp | ₹6,10,000 | 3 |
| JD 5105 | 40 hp | ₹7,20,000 | 2 |
| JD 5050D | 50 hp | ₹8,50,000 | 4 |

*Source: **search_inventory** tool.*

> Finished chain.

Q: Which tractors do you have in stock under 9 lakh rupees?
A: **Tractors in stock ≤ ₹9  lakh (as of today)**  

| Model | Horsepower | On‑roa

,id,category,tools_called,expected_tools,tool_selection_ok,steps,error_handled_ok,latency_s
0,S1,single tool (database),search_inventory,search_inventory,True,1,True,3.91
1,S2,multi-step (database then calculator),"search_inventory, calculate_purchase_plan","search_inventory, calculate_purchase_plan",True,2,True,8.54
2,S3,memory follow-up,calculate_purchase_plan,calculate_purchase_plan,True,1,True,7.17
3,S4,external API (weather),get_weather_forecast,get_weather_forecast,True,1,True,11.31
4,S5,database (service history),search_service_records,search_service_records,True,1,True,3.37
5,S6,calculator (fuel),estimate_operating_cost,estimate_operating_cost,True,1,True,2.91
6,S7,tool failure (expected),get_mandi_price,get_mandi_price,True,1,True,3.24
7,S8,scheme rules (RAG tool),"lookup_scheme_rules, lookup_scheme_rules",lookup_scheme_rules,True,2,True,4.42
8,S9,out of scope (should refuse),(none),(none),True,0,True,2.28


## 10. Failure case: tool returns an error
The mandi-price tool has no API key configured, so it returns `TOOL_ERROR`. The agent must tell the user instead of inventing a price.

In [28]:
row = results[results['id'] == 'S7'].iloc[0]
print(row['question']); print('\ntools called:', row['tools_called']); print('\nanswer:', row['answer']); print('\ndata gaps:', row['data_gaps'])

What is the mandi price of paddy in Chhattisgarh right now?

tools called: get_mandi_price

answer: I’m unable to retrieve the current mandi price for paddy in Chhattisgarh because the data source isn’t available at the moment. If you have an expected selling price or need any other assistance (e.g., subsidy calculations, machine selection), please let me know.

data gaps: Mandi price data unavailable due to missing API key for the data.gov.in service.


## 11. Demo UI

In [29]:

import os
import sys
import uuid

import gradio as gr

from kisanmitra.agent import build_conversational_agent, get_llm, run  # noqa: E402


SESSION = str(uuid.uuid4())


def respond(message, history):
    out = run(chain, llm, message, session_id=SESSION, verbose=False)
    card = out["card"]
    text = out["result"]["output"]
    if card.key_figures:
        text += "\n\n**Key figures:** " + "; ".join(card.key_figures)
    text += f"\n\n**Tools used:** {', '.join(card.tools_used) or 'none'} · **Confidence:** {card.confidence}"
    if card.data_gaps:
        text += f"\n\n⚠️ {card.data_gaps}"
    return text


demo = gr.ChatInterface(
    respond,
    title="KisanMitra 🚜 — dealership operations agent",
    description=f"Tools available: {', '.join(t.name for t in tools)}",
    examples=["Which tractors under 9 lakh are in stock?",
              "EMI for the JD 5050D with 40% subsidy, 1 lakh down payment, 5 years?",
              "Is the weather near Dhamtari good for spraying in the next 3 days?"],
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8cf320fa29180eb7a5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
